# Project 4: LASSO Regression & Feature Selection

## Introduction

**LASSO** (Least Absolute Shrinkage and Selection Operator) is a regression method that performs both prediction and automatic feature selection through L1 regularization.

### The LASSO Problem

$$\min_{\beta} \frac{1}{2n}\|y - X\beta\|_2^2 + \lambda\|\beta\|_1$$

Where:
- $\|\beta\|_1 = \sum_i |\beta_i|$ (L1 norm)
- $\lambda \geq 0$ controls regularization strength

### Why L1 Regularization?

**L1 penalty** ($|\beta_i|$) drives coefficients to exactly zero → **automatic feature selection**!

Compare with **Ridge (L2)**:
- Ridge: $\lambda\|\beta\|_2^2 = \lambda\sum_i \beta_i^2$ → small but non-zero coefficients
- LASSO: $\lambda\|\beta\|_1 = \lambda\sum_i |\beta_i|$ → exact zeros!

### Learning Objectives

1. Understand why L1 causes sparsity
2. Implement LASSO using CVXPy
3. Apply to high-dimensional data
4. Compare LASSO vs Ridge vs OLS
5. Use LASSO for feature selection

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cvxpy as cp
from sklearn.linear_model import Lasso, Ridge, LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## Part 1: Why L1 Causes Sparsity - Geometric Intuition

In [ ]:
# Visualize constraint regions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# L1 constraint region (diamond)
theta = np.linspace(0, 2*np.pi, 100)
x_l1 = np.abs(np.cos(theta))
y_l1 = 1 - x_l1  # L1 ball: |x| + |y| <= 1
x_l1 = x_l1 * np.sign(np.cos(theta))
y_l1 = y_l1 * np.sign(np.sin(theta))

ax1.fill(x_l1, y_l1, alpha=0.3, color='blue', label='L1 constraint')
ax1.plot(x_l1, y_l1, 'b-', linewidth=2)

# Add elliptical contours (RSS)
for level in [0.5, 1.0, 1.5, 2.0]:
    circle = plt.Circle((0.7, 0.7), level*0.4, fill=False, 
                        color='red', linestyle='--', alpha=0.6)
    ax1.add_patch(circle)

ax1.plot(0, 0, 'go', markersize=15, label='Sparse solution (on axis)')
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_xlabel(r'$\beta_1$', fontsize=13)
ax1.set_ylabel(r'$\beta_2$', fontsize=13)
ax1.set_title('L1 Regularization (LASSO)', fontsize=14, fontweight='bold')
ax1.axhline(0, color='k', linewidth=0.5)
ax1.axvline(0, color='k', linewidth=0.5)
ax1.legend(loc='upper right')
ax1.grid(alpha=0.3)
ax1.set_aspect('equal')

# L2 constraint region (circle)
theta = np.linspace(0, 2*np.pi, 100)
x_l2 = np.cos(theta)
y_l2 = np.sin(theta)

ax2.fill(x_l2, y_l2, alpha=0.3, color='blue', label='L2 constraint')
ax2.plot(x_l2, y_l2, 'b-', linewidth=2)

# Add elliptical contours
for level in [0.5, 1.0, 1.5, 2.0]:
    circle = plt.Circle((0.7, 0.7), level*0.4, fill=False, 
                        color='red', linestyle='--', alpha=0.6)
    ax2.add_patch(circle)

ax2.plot(0.5, 0.5, 'go', markersize=15, label='Non-sparse solution')
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_xlabel(r'$\beta_1$', fontsize=13)
ax2.set_ylabel(r'$\beta_2$', fontsize=13)
ax2.set_title('L2 Regularization (Ridge)', fontsize=14, fontweight='bold')
ax2.axhline(0, color='k', linewidth=0.5)
ax2.axvline(0, color='k', linewidth=0.5)
ax2.legend(loc='upper right')
ax2.grid(alpha=0.3)
ax2.set_aspect('equal')

plt.suptitle('Why L1 Causes Sparsity: Diamond vs Circle', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key Insight: L1 constraint (diamond) has corners on axes → solution hits corner → sparse!")
print("L2 constraint (circle) is smooth → solution rarely exactly on axis → not sparse.")

## Part 2: Implement LASSO with CVXPy

In [ ]:
def lasso_cvxpy(X, y, lambda_reg):
    """
    Solve LASSO using CVXPy.
    
    min (1/2n)||y - X*beta||_2^2 + lambda*||beta||_1
    """
    n, p = X.shape
    
    # Define optimization variable
    beta = cp.Variable(p)
    
    # Define objective
    objective = cp.Minimize(
        (1/(2*n)) * cp.sum_squares(y - X @ beta) + lambda_reg * cp.norm(beta, 1)
    )
    
    # Solve
    problem = cp.Problem(objective)
    problem.solve()
    
    return beta.value, problem.value

def ridge_cvxpy(X, y, lambda_reg):
    """
    Solve Ridge using CVXPy for comparison.
    """
    n, p = X.shape
    beta = cp.Variable(p)
    
    objective = cp.Minimize(
        (1/(2*n)) * cp.sum_squares(y - X @ beta) + lambda_reg * cp.sum_squares(beta)
    )
    
    problem = cp.Problem(objective)
    problem.solve()
    
    return beta.value, problem.value

print("✓ LASSO and Ridge implemented with CVXPy!")

## Part 3: Generate Sparse Ground Truth Data

In [ ]:
# Generate data with sparse true coefficients
n_samples = 200
n_features = 50
n_informative = 10  # Only 10 features are actually relevant

X, y, true_coef = make_regression(
    n_samples=n_samples,
    n_features=n_features,
    n_informative=n_informative,
    noise=10.0,
    coef=True,
    random_state=42
)

# Standardize
scaler_X = StandardScaler()
X = scaler_X.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")
print(f"\nTrue coefficient sparsity: {np.sum(true_coef == 0)} out of {len(true_coef)} are zero")

# Visualize true coefficients
plt.figure(figsize=(12, 4))
plt.stem(range(len(true_coef)), true_coef, basefmt=' ')
plt.xlabel('Feature Index', fontsize=12)
plt.ylabel('True Coefficient Value', fontsize=12)
plt.title('True Sparse Coefficients (Ground Truth)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Compare LASSO vs Ridge vs OLS

In [ ]:
# Choose regularization parameter
lambda_reg = 1.0

# Fit models
print("Fitting models...\n")

# OLS (no regularization)
ols = LinearRegression()
ols.fit(X_train, y_train)
beta_ols = ols.coef_
y_pred_ols = ols.predict(X_test)
mse_ols = mean_squared_error(y_test, y_pred_ols)

# LASSO
beta_lasso, _ = lasso_cvxpy(X_train, y_train, lambda_reg)
y_pred_lasso = X_test @ beta_lasso
mse_lasso = mean_squared_error(y_test, y_pred_lasso)

# Ridge
beta_ridge, _ = ridge_cvxpy(X_train, y_train, lambda_reg)
y_pred_ridge = X_test @ beta_ridge
mse_ridge = mean_squared_error(y_test, y_pred_ridge)

# Compare sparsity
print("=== Sparsity Comparison ===")
print(f"True coefficients: {np.sum(np.abs(true_coef) > 0.1)} non-zero")
print(f"OLS coefficients: {np.sum(np.abs(beta_ols) > 0.1)} non-zero")
print(f"Ridge coefficients: {np.sum(np.abs(beta_ridge) > 0.1)} non-zero")
print(f"LASSO coefficients: {np.sum(np.abs(beta_lasso) > 0.1)} non-zero")

print("\n=== Exact Zeros ===")
print(f"True: {np.sum(true_coef == 0)} exact zeros")
print(f"OLS: {np.sum(beta_ols == 0)} exact zeros")
print(f"Ridge: {np.sum(np.abs(beta_ridge) < 1e-10)} exact zeros")
print(f"LASSO: {np.sum(np.abs(beta_lasso) < 1e-6)} exact zeros")

print("\n=== Test MSE ===")
print(f"OLS:   {mse_ols:.2f}")
print(f"Ridge: {mse_ridge:.2f}")
print(f"LASSO: {mse_lasso:.2f}")

## Part 5: Visualize Coefficient Paths

In [ ]:
# Compare coefficients
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# True coefficients
axes[0,0].stem(range(len(true_coef)), true_coef, basefmt=' ')
axes[0,0].set_title('True Coefficients', fontsize=13, fontweight='bold')
axes[0,0].set_xlabel('Feature Index')
axes[0,0].set_ylabel('Coefficient Value')
axes[0,0].grid(alpha=0.3)

# OLS
axes[0,1].stem(range(len(beta_ols)), beta_ols, basefmt=' ', linefmt='C1-', markerfmt='C1o')
axes[0,1].set_title(f'OLS (Test MSE: {mse_ols:.2f})', fontsize=13, fontweight='bold')
axes[0,1].set_xlabel('Feature Index')
axes[0,1].set_ylabel('Coefficient Value')
axes[0,1].grid(alpha=0.3)

# Ridge
axes[1,0].stem(range(len(beta_ridge)), beta_ridge, basefmt=' ', linefmt='C2-', markerfmt='C2o')
axes[1,0].set_title(f'Ridge (λ={lambda_reg}, Test MSE: {mse_ridge:.2f})', fontsize=13, fontweight='bold')
axes[1,0].set_xlabel('Feature Index')
axes[1,0].set_ylabel('Coefficient Value')
axes[1,0].grid(alpha=0.3)

# LASSO
axes[1,1].stem(range(len(beta_lasso)), beta_lasso, basefmt=' ', linefmt='C3-', markerfmt='C3o')
axes[1,1].set_title(f'LASSO (λ={lambda_reg}, Test MSE: {mse_lasso:.2f})', fontsize=13, fontweight='bold')
axes[1,1].set_xlabel('Feature Index')
axes[1,1].set_ylabel('Coefficient Value')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 6: Regularization Path

In [ ]:
# Compute LASSO solution for different lambda values
lambdas = np.logspace(-2, 2, 50)
coefs_lasso = []
coefs_ridge = []
mse_train_lasso = []
mse_test_lasso = []
mse_train_ridge = []
mse_test_ridge = []
sparsity = []

for lam in lambdas:
    # LASSO
    beta_l, _ = lasso_cvxpy(X_train, y_train, lam)
    coefs_lasso.append(beta_l)
    mse_train_lasso.append(mean_squared_error(y_train, X_train @ beta_l))
    mse_test_lasso.append(mean_squared_error(y_test, X_test @ beta_l))
    sparsity.append(np.sum(np.abs(beta_l) > 1e-6))
    
    # Ridge
    beta_r, _ = ridge_cvxpy(X_train, y_train, lam)
    coefs_ridge.append(beta_r)
    mse_train_ridge.append(mean_squared_error(y_train, X_train @ beta_r))
    mse_test_ridge.append(mean_squared_error(y_test, X_test @ beta_r))

coefs_lasso = np.array(coefs_lasso)
coefs_ridge = np.array(coefs_ridge)

# Plot regularization path
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# LASSO path
for i in range(min(10, n_features)):  # Plot first 10 features
    axes[0,0].plot(lambdas, coefs_lasso[:, i], linewidth=2, alpha=0.7)
axes[0,0].set_xscale('log')
axes[0,0].set_xlabel('λ (log scale)', fontsize=12)
axes[0,0].set_ylabel('Coefficient Value', fontsize=12)
axes[0,0].set_title('LASSO Regularization Path', fontsize=13, fontweight='bold')
axes[0,0].grid(alpha=0.3)

# Ridge path
for i in range(min(10, n_features)):
    axes[0,1].plot(lambdas, coefs_ridge[:, i], linewidth=2, alpha=0.7)
axes[0,1].set_xscale('log')
axes[0,1].set_xlabel('λ (log scale)', fontsize=12)
axes[0,1].set_ylabel('Coefficient Value', fontsize=12)
axes[0,1].set_title('Ridge Regularization Path', fontsize=13, fontweight='bold')
axes[0,1].grid(alpha=0.3)

# MSE curves
axes[1,0].plot(lambdas, mse_train_lasso, 'o-', label='LASSO Train', linewidth=2)
axes[1,0].plot(lambdas, mse_test_lasso, 's-', label='LASSO Test', linewidth=2)
axes[1,0].plot(lambdas, mse_train_ridge, 'o--', label='Ridge Train', linewidth=2, alpha=0.7)
axes[1,0].plot(lambdas, mse_test_ridge, 's--', label='Ridge Test', linewidth=2, alpha=0.7)
axes[1,0].set_xscale('log')
axes[1,0].set_xlabel('λ (log scale)', fontsize=12)
axes[1,0].set_ylabel('MSE', fontsize=12)
axes[1,0].set_title('Train/Test MSE vs Regularization', fontsize=13, fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

# Sparsity
axes[1,1].plot(lambdas, sparsity, 'o-', linewidth=2, color='purple')
axes[1,1].set_xscale('log')
axes[1,1].set_xlabel('λ (log scale)', fontsize=12)
axes[1,1].set_ylabel('Number of Non-zero Coefficients', fontsize=12)
axes[1,1].set_title('LASSO Sparsity vs λ', fontsize=13, fontweight='bold')
axes[1,1].axhline(n_informative, color='red', linestyle='--', label=f'True: {n_informative}')
axes[1,1].legend()
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Find best lambda
best_idx = np.argmin(mse_test_lasso)
best_lambda = lambdas[best_idx]
print(f"\nBest λ for LASSO: {best_lambda:.4f}")
print(f"Test MSE at best λ: {mse_test_lasso[best_idx]:.2f}")
print(f"Sparsity at best λ: {sparsity[best_idx]} non-zero coefficients")

## Part 7: Feature Selection with LASSO

In [ ]:
# Use LASSO for feature selection
selected_lambda = best_lambda
beta_selected, _ = lasso_cvxpy(X_train, y_train, selected_lambda)

# Identify selected features
selected_features = np.where(np.abs(beta_selected) > 1e-6)[0]
true_features = np.where(np.abs(true_coef) > 0.1)[0]

print(f"\n=== Feature Selection Results ===")
print(f"True relevant features: {list(true_features)}")
print(f"LASSO selected features: {list(selected_features)}")
print(f"\nTrue positives: {len(set(selected_features) & set(true_features))}")
print(f"False positives: {len(set(selected_features) - set(true_features))}")
print(f"False negatives: {len(set(true_features) - set(selected_features))}")

# Visualize feature selection
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Selected vs True
feature_mask = np.zeros(n_features)
feature_mask[true_features] = 1  # True features
feature_mask[selected_features] += 2  # Selected features
# Now: 0 = neither, 1 = true only, 2 = selected only, 3 = both

colors = ['white', 'blue', 'red', 'green']
labels = ['Neither', 'True only', 'Selected only', 'Both (correct)']
cmap = plt.matplotlib.colors.ListedColormap(colors)

im = ax1.imshow(feature_mask.reshape(1, -1), cmap=cmap, aspect='auto')
ax1.set_yticks([])
ax1.set_xlabel('Feature Index', fontsize=12)
ax1.set_title('Feature Selection Comparison', fontsize=13, fontweight='bold')

# Create legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colors[i], label=labels[i]) for i in range(4)]
ax1.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.3, 1))

# Coefficient comparison
x = np.arange(len(selected_features))
width = 0.35

true_vals = [true_coef[i] if i in true_features else 0 for i in selected_features]
lasso_vals = [beta_selected[i] for i in selected_features]

ax2.bar(x - width/2, true_vals, width, label='True', alpha=0.8)
ax2.bar(x + width/2, lasso_vals, width, label='LASSO', alpha=0.8)
ax2.set_xlabel('Selected Feature Index', fontsize=12)
ax2.set_ylabel('Coefficient Value', fontsize=12)
ax2.set_title('LASSO vs True Coefficients (Selected Features)', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(selected_features, rotation=45)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

### What We Learned

1. **L1 regularization** ($\|\beta\|_1$) → **sparsity** (exact zeros)
2. **L2 regularization** ($\|\beta\|_2^2$) → small but non-zero coefficients
3. **LASSO = automatic feature selection** + prediction
4. **λ controls sparsity**: larger λ → more zeros
5. **CVXPy** makes solving LASSO easy

### When to Use LASSO vs Ridge

**Use LASSO when:**
- You believe many features are irrelevant (sparse truth)
- You want interpretability (few features)
- You want automatic feature selection
- p >> n (more features than samples)

**Use Ridge when:**
- All features contribute (dense truth)
- Features are highly correlated (LASSO picks one arbitrarily)
- You care about prediction, not interpretation

**Use Elastic Net when:**
- Mix of both: $\lambda_1\|\beta\|_1 + \lambda_2\|\beta\|_2^2$
- Correlated features + want sparsity

### Real-World Applications

- **Genomics**: Select relevant genes from thousands
- **Finance**: Portfolio optimization with sparse holdings
- **NLP**: Text classification with sparse features
- **Computer Vision**: Compressed sensing, image recovery

### Connection to ML

- **Deep Learning**: L1 regularization for neural network pruning
- **Feature Engineering**: Automated feature selection before training
- **Compressed Sensing**: Recover signals from few measurements (Project 7!)

### Next Steps

- **Project 5**: Logistic Regression with gradient descent
- **Project 6**: Portfolio optimization with constraints
- **Project 7**: Compressed sensing (LASSO's killer app!)

---

**Excellent work!** You now understand why L1 regularization is magical for feature selection! ✨